In [8]:
import requests
from bs4 import BeautifulSoup
import regex as re

          
url_produto = 'https://www.farmaponte.com.br/nortriptilina-50-30cap-ran/p'
response_dados = requests.get(url_produto)
soup_dados = BeautifulSoup(response_dados.text, 'html.parser')
nome_produto = soup_dados.find('h1', class_='name').text
scripts = soup_dados.find_all('script', type='text/javascript')

# Encontrar o script que contém 'dataItem'
data_script = None
for script in scripts:
    if script.string and 'dataItem' in script.string:
        data_script = script.string
        break

# Passo 3: Extrair as informações do script
if data_script:
    # Expressões regulares para extrair o preço e desconto
    price_pattern = re.compile(r'"price":([\d.]+)')
    discount_pattern = re.compile(r'"discount":([\d.]+)')
    
    # Extraindo os valores
    price_match = price_pattern.search(data_script)
    discount_match = discount_pattern.search(data_script)
    
    if price_match and discount_match:
        price = float(price_match.group(1))
        discount = float(discount_match.group(1))
        print(price)
        print(discount)
    else:
        print("Preço ou desconto não encontrado.")
else:
    print("Script com os dados não encontrado.")


56.1
10.1


In [2]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import regex as re

url = 'https://www.farmaponte.com.br/s/farmaponte/sitemap-categories-1.xml'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'xml')

# Extraindo os links para cada categoria
links_categorias = soup.find_all('loc')
link_med_geral = False
links_produtos = []
for link in links_categorias:
    if '/medicamentos' in link.text:
        if not link_med_geral:
            link_med_geral = True
        else:
            pagina = 1 
            while True:
                link_pagina = f"{link.text}?p={pagina}"
                response_produtos = requests.get(link_pagina) # Extraindo links de cada produto por categoria
                soup_produtos = BeautifulSoup(response_produtos.text, 'html.parser')
                titulo = soup_produtos.find_all('h2', class_='title')
                if not titulo:
                    break
                for cada in titulo:
                    link_produto = cada.find('a', href=True)
                    if link_produto:
                        links_produtos.append(link_produto['href'])
                pagina += 1



def extracao_final(links_produtos):
    nomes = []
    precos_sem_desconto = []
    precos_pix = []
    porcentagens_desconto = []
    valores_descontos = []
    marcas = []
    eans = []
    
    for i in range(len(links_produtos)):            
        url_produto = f'https://www.farmaponte.com.br/{links_produtos[i]}'
        response_dados = requests.get(url_produto)
        soup_dados = BeautifulSoup(response_dados.text, 'html.parser')
        nome_produto = soup_dados.find('h1', class_='name').text
        scripts = soup_dados.find_all('script', type='text/javascript')

        # Encontrar o script que contém 'dataItem'
        data_script = None
        for script in scripts:
            if script.string and 'dataItem' in script.string:
                data_script = script.string
                break

        # Passo 3: Extrair as informações do script
        if data_script:
            # Expressões regulares para extrair o preço e desconto
            price_pattern = re.compile(r'"price":([\d.]+)')
            discount_pattern = re.compile(r'"discount":([\d.]+)')
            
            # Extraindo os valores
            price_match = price_pattern.search(data_script)
            discount_match = discount_pattern.search(data_script)

        if price_match:
            price = float(price_match.group(1))
        else:
            price = 'Preço indisponível'
        
        if discount_match:
            discount = float(discount_match.group(1))
        else:
            discount = 0

        # Pegar preço do pix
        preco_pix = soup_dados.find('div', class_='pix-price')
        if preco_pix:
            preco_pix = preco_pix.text.strip().replace('no pix', '')
        else: 
            preco_pix = price

        # Pegar porcentagem de desconto
        porcentagem_desconto = soup_dados.find('span', class_='discount')
        if porcentagem_desconto:
            porcentagem_desconto = porcentagem_desconto.text.replace('off', '')
        else:
            porcentagem_desconto = '0%'

        # Pegar EAN
        ean = soup_dados.find('meta', itemprop='gtin13')
        if ean and 'content' in ean.attrs:
            eans.append(ean['content'].strip())
        else:
            eans.append("EAN não informado")

        aux_marca = soup_dados.find('meta', {'itemprop': 'brand'})
        if aux_marca:    
            marca_produto = aux_marca['content']
        elif 'adv' in nome_produto:
            marca_produto = 'ADV Farma'
        elif 'Abbott' in nome_produto:
            marca_produto = 'Abbott do Brasil'
        elif 'Addera' in nome_produto:
            marca_produto = 'Addera'
        elif 'Allergan' in nome_produto:
            marca_produto = 'Allergan'
        elif 'AstraZeneca' in nome_produto:
            marca_produto = 'Astrazeneca'
        elif 'Ache' in nome_produto:
            marca_produto = 'Aché'
        elif 'Biolab Genérico' in nome_produto:
            marca_produto = 'Biolab Genéricos'
        elif 'Biosintética' in nome_produto:
            marca_produto = 'Biosintética - Aché'
        elif 'Catarinense' in nome_produto:
            marca_produto = 'Catarinense Pharma'
        elif 'Cimed' in nome_produto:
            marca_produto = 'Cimed'
        elif 'Cosmed' in nome_produto:
            marca_produto = 'Cosmed'
        elif 'Diffucap' in nome_produto:
            marca_produto = 'Diffucap Chemobras'
        elif 'EMS' in nome_produto or 'Ems' in nome_produto or 'ems' in nome_produto:
            marca_produto = 'EMS'
        elif 'Sigma' in nome_produto:
            marca_produto = 'EMS Sigma Pharma'
        elif 'Eliquis' in nome_produto:
            marca_produto = 'Pfizer'
        elif "Eur" in nome_produto or 'Eurofarma' in nome_produto:
            marca_produto = 'Eurofarma'
        elif 'Farmoquímica' in nome_produto:
            marca_produto = 'Farmoquímica'
        elif 'GSK' in nome_produto or 'Gsk' in nome_produto:
            marca_produto = 'GSK'
        elif 'Geolab' in nome_produto:
            marca_produto = 'Geolab'
        elif 'Germed' in nome_produto or 'germed' in nome_produto:
            marca_produto = 'Germed Pharma'
        elif 'Glenmark' in nome_produto:
            marca_produto = 'Glenmark'
        elif 'Grunenthal' in nome_produto:
            marca_produto = 'Grünenthal'
        elif 'Legrand' in nome_produto:
            marca_produto = 'Legrand'
        elif 'Libbs' in nome_produto:
            marca_produto = 'Libbs'
        elif 'Merck' in nome_produto:
            marca_produto = 'Merck'
        elif 'Med' in nome_produto or 'med' in nome_produto:
            marca_produto = 'Medley'
        elif 'neo' in nome_produto or 'Neo' in nome_produto:
            marca_produto = "Neo Química"
        elif 'Novartis' in nome_produto:
            marca_produto = 'Novartis'
        elif 'Prati' in nome_produto or 'PRATI' in nome_produto:
            marca_produto = 'Prati-Donaduzzi'
        elif 'Ranbaxy' in nome_produto:
            marca_produto = 'Ranbaxy'
        elif 'Sandoz' in nome_produto:
            marca_produto = 'Sandoz'
        elif 'Sanofi' in nome_produto:
            marca_produto = 'Sanofi'
        elif 'Servier' in nome_produto:
            marca_produto = 'Servier'
        elif 'Supera' in nome_produto:
            marca_produto = 'Supera'
        elif 'Teuto' in nome_produto:
            marca_produto = 'Teuto'
        elif 'TORRENT' in nome_produto or 'Torrent' in nome_produto:
            marca_produto = 'Torrent'
        elif 'União' in nome_produto or 'Uniao' in nome_produto:
            marca_produto = 'União Química'
        else:
            marca_produto = "Não especificada"

        nomes.append(nome_produto) 
        precos_sem_desconto.append(price)
        precos_pix.append(preco_pix)
        porcentagens_desconto.append(porcentagem_desconto)
        marcas.append(marca_produto)
        valores_descontos.append(discount)

    return nomes, precos_sem_desconto, precos_pix, valores_descontos, porcentagens_desconto, marcas, eans

def chunk_list(list, n):
    for i in range(0, len(list), n):
        yield list[i:i + n]
chunks = list(chunk_list(links_produtos, 10))

todos_nomes = []
todos_precos_sem_desconto = []
todos_precos_pix = []
todos_valores_descontos = []
todas_porcentagens_desconto = []
todas_marcas = []
todos_ean = []

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(extracao_final, chunk) for chunk in chunks]
    for future in as_completed(futures):
        nomes, precos_sem_desconto, precos_pix, valores_descontos, porcentagens_desconto, marcas, eans = future.result()
        todos_nomes.extend(nomes)
        todos_precos_sem_desconto.extend(precos_sem_desconto)
        todos_precos_pix.extend(precos_pix)
        todos_valores_descontos.extend(valores_descontos)
        todas_porcentagens_desconto.extend(porcentagens_desconto)
        todas_marcas.extend(marcas)
        todos_ean.extend(eans)

df = pd.DataFrame({
    'Nome': todos_nomes,
    'Marca': todas_marcas,
    'EAN' : todos_ean,
    'Preço sem desconto': todos_precos_sem_desconto,
    'Preço no pix': todos_precos_pix,
    'Valor do desconto': todos_valores_descontos,
    'Porcentagem desconto': todas_porcentagens_desconto
    })             

df



In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4494 entries, 0 to 4493
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Nome                  4494 non-null   object 
 1   Marca                 4494 non-null   object 
 2   EAN                   4494 non-null   object 
 3   Preço sem desconto    4494 non-null   float64
 4   Preço no pix          4494 non-null   object 
 5   Valor do desconto     4494 non-null   object 
 6   Porcentagem desconto  4494 non-null   object 
dtypes: float64(1), object(6)
memory usage: 245.9+ KB


In [18]:
df.to_csv('farmaponte.csv', index = False) 
df.to_excel('farmaponte.xlsx', index = False) 

In [13]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

reqUrl = "https://www.farmaponte.com.br/saude/medicamentos/"

headersList = {
 "Accept": "*/*",
 "User-Agent": "Thunder Client (https://www.thunderclient.com)" 
}

payload = ""

response = requests.request("GET", reqUrl, data=payload,  headers=headersList)

soup = BeautifulSoup(response.text, 'html.parser')

filtro_marca = soup.find('ul', id='nav-sidebar-marca-2')

small_tags = filtro_marca.find_all('small')

small_values = [int(small_tag.text) for small_tag in small_tags]

total_sum = sum(small_values)

print(total_sum)




3416
